# Tech Challenge Fase 2 - Classificação da Qualidade de Vinhos

## 03. Modelagem

Terceiro notebook do projeto. Recebe a base gerada no `02_preprocessamento.ipynb` (já com as variáveis derivadas), separa treino e teste, e treina os modelos de classificação. A avaliação em si (métricas, importância de variáveis, conclusões) fica para o `04_avaliacao.ipynb`.

## 1. Instalação e importação das bibliotecas

In [1]:
# Se alguma biblioteca não existir no Colab, descomente a linha abaixo:
# !pip install -q pandas numpy scikit-learn joblib

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs("data/processed", exist_ok=True)
os.makedirs("modelos", exist_ok=True)

## 2. Carregar a base gerada no notebook anterior

In [2]:
features_path = "data/processed/wine_quality_features.csv"

if not os.path.exists(features_path):
    try:
        from google.colab import files
        print("Faça upload do arquivo wine_quality_features.csv gerado no notebook 02_preprocessamento")
        uploaded = files.upload()
        uploaded_name = list(uploaded.keys())[0]
        os.replace(uploaded_name, features_path)
    except Exception:
        raise FileNotFoundError("Arquivo wine_quality_features.csv não encontrado. Rode o notebook 02_preprocessamento primeiro.")

df_model = pd.read_csv(features_path)
print("Base carregada!")
print("Linhas:", df_model.shape[0])
print("Colunas:", df_model.shape[1])
df_model.head()

Base carregada!
Linhas: 1018
Colunas: 17


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id,high_quality,acidez_total,razao_so2_livre,alcool_por_densidade
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0,0,8.10,0.323529,9.420726
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1,0,8.68,0.373134,9.831461
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2,0,8.56,0.277778,9.829488
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3,0,11.48,0.283333,9.819639
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,5,0,8.06,0.325000,9.420726


## 3. Divisão treino e teste

In [3]:
id_cols = ["Id"] if "Id" in df_model.columns else []
target_cols = ["quality", "high_quality"]
feature_cols = [c for c in df_model.columns if c not in id_cols + target_cols]

X = df_model[feature_cols].copy()
y = df_model["high_quality"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("Distribuição treino:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("Distribuição teste:")
print(y_test.value_counts(normalize=True).sort_index().round(3))

Treino: (763, 14)
Teste: (255, 14)
Distribuição treino:
high_quality
0    0.865
1    0.135
Name: proportion, dtype: float64
Distribuição teste:
high_quality
0    0.867
1    0.133
Name: proportion, dtype: float64


A divisão é estratificada pela classe (`stratify=y`), garantindo que a proporção de vinhos de alta qualidade seja a mesma no treino e no teste. Isso evita, por exemplo, que o conjunto de teste fique sem nenhum exemplo da classe minoritária por azar do sorteio. O `RANDOM_STATE` fixo garante que essa divisão seja sempre a mesma se o notebook for rodado de novo.

## 4. Definição e treino dos modelos

In [4]:
models = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, random_state=RANDOM_STATE,
        class_weight="balanced_subsample", min_samples_leaf=2, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print("Modelo treinado:", name)

Modelo treinado: Regressão Logística


Modelo treinado: Random Forest


Modelo treinado: Gradient Boosting


Os três modelos usam a mesma `RANDOM_STATE` e o mesmo `class_weight="balanced"` (ou equivalente), já que a classe de alta qualidade é minoritária na base. A Regressão Logística entra padronizada dentro de um `Pipeline`, os dois modelos de árvore não precisam de padronização, como discutido no notebook de pré-processamento.

## 5. Salvar modelos e dados para a próxima etapa

In [5]:
import json as _json_mod

nomes_arquivos = {}
for name, model in models.items():
    nome_arquivo = f"modelo_{len(nomes_arquivos)}"
    joblib.dump(model, f"modelos/{nome_arquivo}.joblib")
    nomes_arquivos[nome_arquivo] = name

with open("modelos/nomes_arquivos.json", "w") as f:
    _json_mod.dump(nomes_arquivos, f, ensure_ascii=False)

X_train.to_csv("modelos/X_train.csv", index=False)
X_test.to_csv("modelos/X_test.csv", index=False)
y_train.to_csv("modelos/y_train.csv", index=False)
y_test.to_csv("modelos/y_test.csv", index=False)
X.to_csv("modelos/X_full.csv", index=False)
y.to_csv("modelos/y_full.csv", index=False)

with open("modelos/feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Arquivos salvos em modelos/:")
for f in sorted(os.listdir("modelos")):
    print("-", f)

Arquivos salvos em modelos/:
- X_full.csv
- X_test.csv
- X_train.csv
- feature_cols.json
- modelo_0.joblib
- modelo_1.joblib
- modelo_2.joblib
- nomes_arquivos.json
- y_full.csv
- y_test.csv
- y_train.csv


In [6]:
import shutil

shutil.make_archive("pacote_modelagem", "zip", "modelos")

try:
    from google.colab import files
    files.download("pacote_modelagem.zip")
except Exception:
    pass

O arquivo `pacote_modelagem.zip` baixado aqui contém os três modelos treinados e os conjuntos de treino e teste. Ele é a entrada do próximo notebook (`04_avaliacao.ipynb`), onde os modelos são comparados, o melhor é escolhido, e os resultados finais são interpretados.